# Book Recommender Project

#### Personalized book suggestions using data, NLP, and Streamlit

Books Scrapped from [goodreads](https://www.goodreads.com/book/popular_by_date/2019).
* Years: 1975-2024

Books API from [openlibrary](https://openlibrary.org/developers/api). 
* Years: 1900-1975

Developed by: Bryan Calderon

Extracting the 10 most popular books per year!


## **API source**

In [ ]:
# Impoting libraries
import requests
import pandas as pd
import numpy as np
import time
import os

In [24]:
base_api_url = "https://openlibrary.org/search.json"

start_year = 1900
end_year = 1975
books_per_year = 10

api_books = []
failed_years = []

* Defining functions for the extraction of the information from the API 

In [25]:
def clean_list(value):
    """
    Converts list values into comma-separated strings.
    """
    if isinstance(value, list):
        return ", ".join([str(v) for v in value])
    return value

In [26]:
def get_cover_url(cover_id):
    """
    Creates an Open Library cover image URL from cover_i.
    """
    if pd.isna(cover_id) or cover_id is None:
        return None
    
    return f"https://covers.openlibrary.org/b/id/{int(cover_id)}-M.jpg"

In [27]:
def get_openlibrary_url(key):
    """
    Creates an Open Library work URL from a work key.
    """
    if key is None:
        return None
    
    return f"https://openlibrary.org{key}"

In [28]:
def safe_openlibrary_request(params, retries=3, sleep_time=2):
    """
    Makes a safe request to Open Library API.
    """
    for attempt in range(retries):
        try:
            response = requests.get(base_api_url, params=params, timeout=20)
            
            if response.status_code == 200:
                return response.json()
            
            print(f"Error {response.status_code}: {response.text[:200]}")
            time.sleep(sleep_time)
        
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed.")
            print(e)
            time.sleep(sleep_time)
    
    return None

In [ ]:
for year in range(start_year, end_year + 1):
    print(f"Collecting Open Library books for year {year}...")
    
    params = {
        "q": f"subject:fiction",
        "first_publish_year": year,
        "language": "eng",
        "limit": 20,
        "fields": (
            "key,title,author_name,first_publish_year,"
            "subject,ratings_average,ratings_count,"
            "edition_count,isbn,cover_i,language,publisher"
        )
    }
    
    data = safe_openlibrary_request(params)
    
    if data is None:
        failed_years.append(year)
        continue
    
    docs = data.get("docs", [])
    
    if len(docs) == 0:
        print(f"No books found for year {year}")
        failed_years.append(year)
        continue
    
    books_added = 0
    
    for doc in docs:
        title = doc.get("title")
        authors = doc.get("author_name", [])
        author = clean_list(authors)
        
        published_year = doc.get("first_publish_year")
        
        # Keep only exact target year
        if published_year != year:
            continue
        
        subjects = doc.get("subject", [])
        genres = clean_list(subjects[:10]) if isinstance(subjects, list) else subjects
        
        isbn_list = doc.get("isbn", [])
        isbn_10_or_13 = isbn_list[0] if isinstance(isbn_list, list) and len(isbn_list) > 0 else None
        
        cover_id = doc.get("cover_i")
        image_url = get_cover_url(cover_id)
        
        book_url = get_openlibrary_url(doc.get("key"))
        
        languages = doc.get("language", [])
        language = clean_list(languages)
        
        publishers = doc.get("publisher", [])
        publisher = clean_list(publishers[:5]) if isinstance(publishers, list) else publishers
        
        api_books.append({
            "source_year": year,
            "rank": None,
            "list_title": None,
            "title": title,
            "author": author,
            "average_rating": doc.get("ratings_average"),
            "ratings_count": doc.get("ratings_count"),
            "reviews_count": None,
            "description": None,
            "genres": genres,
            "pages": None,
            "published_date": str(published_year),
            "language": language,
            "publisher": publisher,
            "isbn": isbn_10_or_13,
            "edition_count": doc.get("edition_count"),
            "book_url": book_url,
            "image_url": image_url,
            "source": "Open Library API"
        })
        
        books_added += 1
        
        if books_added >= books_per_year:
            break
    
    if books_added == 0:
        failed_years.append(year)
    
    time.sleep(1)

NOTE: The extraction of the information from the API openlibrary takes araound 10 min.

In [ ]:
df0 = pd.DataFrame(api_books)
df0.head()

,source_year,rank,list_title,title,author,average_rating,ratings_count,reviews_count,description,genres,pages,published_date,language,publisher,isbn,edition_count,book_url,image_url,source
0,1900,None,None,The Lost World,Arthur Conan Doyle,4.045454,44.0,None,None,"Adventure stories, Atlantis, Dinosaurs, Discov...",None,1900,"ger, eng, spa, cze, chi, ita, rus, fin, fre","Babblebooks, Sound Room Publishers, Incorporat...",1549678922,747,https://openlibrary.org/works/OL262460W,https://covers.openlibrary.org/b/id/8231444-M.jpg,Open Library API
1,1900,None,None,The Clan of the Cave Bear,Jean M. Auel,4.054794,73.0,None,None,"Neanderthals, shamanism, sign language, taboos...",None,1900,"ger, eng, pol, spa, ita, swe, dut, fre, hrv","Brilliance Audio on MP3-CD Lib Ed, Wolfgang Kr...",9788415120032,89,https://openlibrary.org/works/OL2746369W,https://covers.openlibrary.org/b/id/12529015-M...,Open Library API
2,1900,None,None,No country for old men,Cormac McCarthy,4.111111,36.0,None,None,"Fiction, Drug traffic, Sheriffs, Treasure-trov...",None,1900,"eng, rus, fre, pol, spa, tur, kor, ita","GPO ED RANDOM HOUSE MONDADORI, MacMillan, Bran...",9780375406775,44,https://openlibrary.org/works/OL40875W,https://covers.openlibrary.org/b/id/9296899-M.jpg,Open Library API
3,1900,None,None,The Prophet,Kahlil Gibran,4.321429,56.0,None,None,"Poetry, Classics, Literature & Fiction, Religion",None,1900,"ger, heb, guj, eng, tgl, kor, mul, hrv, tur, s...","Patmos-Verlag, Patmos, Ballantine Books, Pmapu...",9501702162,569,https://openlibrary.org/works/OL318900W,https://covers.openlibrary.org/b/id/418324-M.jpg,Open Library API
4,1900,None,None,The Railway Children,Edith Nesbit,3.965517,29.0,None,None,"Brothers and sisters, Children's stories, Clas...",None,1900,"chi, eng, spa, heb, tur, ger, ita","HV Publishign, a division of Homepage Ventures...",9781981610983,985,https://openlibrary.org/works/OL99509W,https://covers.openlibrary.org/b/id/13241123-M...,Open Library API


* Saving data set

In [ ]:
df0.to_csv("data/raw/API_Books.csv", index=False)

* Cleaning data set and variables/columns

In [22]:
df0 = pd.read_csv("data/raw/API_Books.csv")
df0 = df0.drop(columns=["list_title", "rank", "isbn", "edition_count","language", "published_date"]) # dropping columns
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 760 entries, 0 to 759
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   source_year     760 non-null    int64  
 1   title           760 non-null    str    
 2   author          760 non-null    str    
 3   average_rating  745 non-null    float64
 4   ratings_count   745 non-null    float64
 5   reviews_count   0 non-null      float64
 6   description     0 non-null      float64
 7   genres          760 non-null    str    
 8   pages           0 non-null      float64
 9   publisher       760 non-null    str    
 10  book_url        760 non-null    str    
 11  image_url       759 non-null    str    
 12  source          760 non-null    str    
dtypes: float64(5), int64(1), str(7)
memory usage: 385.1 KB


In [ ]:
df0["reviews_count"] = 0  # Adding no reviews = 0
df0["pages"] = None       # Adding None to the pages

# Fixing description-like text because Open Library Search API does not provide descriptions
df0["description"] = (
    df0["title"].fillna("") + " " +
    df0["author"].fillna("") + " " +
    df0["genres"].fillna("") + " " +
    df0["publisher"].fillna("")
)

# Changing data types 
df0 = df0.dropna(subset=["average_rating", "ratings_count", "image_url"]).copy()

# Modifying variables 
df0["average_rating"] = df0["average_rating"].astype(float)
df0["ratings_count"] = df0["ratings_count"].astype(int)
df0["source_year"] = df0["source_year"].astype(int)

In [26]:
df0.isna().sum()

source_year         0
title               0
author              0
average_rating      0
ratings_count       0
reviews_count       0
description         0
genres              0
pages             745
publisher           0
book_url            0
image_url           0
source              0
dtype: int64

In [25]:
df0.duplicated().sum()

np.int64(0)

In [31]:
df0.info()

<class 'pandas.DataFrame'>
Index: 745 entries, 0 to 759
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   source_year     745 non-null    int64  
 1   title           745 non-null    str    
 2   author          745 non-null    str    
 3   average_rating  745 non-null    float64
 4   ratings_count   745 non-null    int64  
 5   reviews_count   745 non-null    int64  
 6   description     745 non-null    str    
 7   genres          745 non-null    str    
 8   pages           0 non-null      object 
 9   publisher       745 non-null    str    
 10  book_url        745 non-null    str    
 11  image_url       745 non-null    str    
 12  source          745 non-null    str    
dtypes: float64(1), int64(3), object(1), str(8)
memory usage: 612.3+ KB


* Saving Clean Dataset

In [ ]:
df0.to_csv("../data/API_Books_clean.csv", index=False) 

In [29]:
df0.to_csv("data/API_Books_clean.csv", index=False)